# Clustering quality (silhouette)

_Machine Learning — more_

**Score each point: tight inside its cluster, far from the next one.**

Clustering has no labels, so how do you know if it is any good?

     The silhouette scores each point on two questions: is it close to its own cluster, and far from the nearest other cluster?

---

This notebook is a practice scaffold for the **Clustering quality (silhouette)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_wine

data = load_wine(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

### Step 1 — Make blobs and pick a value of k to score

We generate 300 points arranged in 3 well-separated blobs, then deliberately *forget* how many there were. The whole point of the silhouette is to recover the right number of clusters **without** knowing the labels. We'll loop k from 2 to 6, and start by setting up a place to remember the best score we've seen.

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 300 points in 3 fairly tight, well-separated blobs.
X, _ = make_blobs(
    n_samples=300,
    centers=3,
    cluster_std=0.8,
    random_state=0,
)

# Track the best k and its score as we sweep.
best_k = None
best_s = -1.0

### Step 2 — Cluster at each k and score with the silhouette

For every candidate k we run KMeans, then compute the **average silhouette** over all points. Each point's silhouette compares how close it sits to its own cluster against how close it sits to the nearest other cluster, so higher is better. We keep whichever k earns the highest average score.

In [ ]:
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    s = silhouette_score(X, labels)             # mean silhouette over points
    print("k=%d  avg silhouette = %.4f" % (k, s))

    if s > best_s:
        best_s = s
        best_k = k

### Step 3 — Read off the winning k

The k with the highest average silhouette is our best guess at the true number of clusters. Because we built the data from 3 blobs, a working silhouette should land on k = 3 — recovering the structure with no labels at all.

In [ ]:
print("best k by silhouette = %d (true number of blobs = 3)" % best_k)

## Visualize the data & results

_How many clusters does real data really have?_

### Step 1 — Load and standardise real wine data

We move to 178 real wines described by 13 chemical measurements, with the true cultivar labels hidden. The features live on very different scales, so we **standardise** them — KMeans uses raw distances, and without scaling the largest-magnitude feature would dominate. This leaves us with a clean feature matrix to cluster.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# REAL data: 178 wines, 13 chemical features, 3 true cultivars (labels hidden).
wn = load_wine()
X = StandardScaler().fit_transform(wn.data)

### Step 2 — Sweep k and record the average silhouette

Just like the toy run, we cluster the wines at each k from 2 to 6 and store the average silhouette. Collecting the values in lists lets us plot the score as a curve against k. The peak of that curve is the silhouette's vote for how many natural groups the wines fall into.

In [ ]:
ks = []
sils = []
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    ks.append(k)
    sils.append(silhouette_score(X, labels))     # mean silhouette over wines

### Step 3 — Plot the silhouette curve

Plotting average silhouette against k shows where clustering quality peaks. If the peak sits at k = 3 it matches the three real cultivars — evidence that the silhouette found the true structure in unlabelled data.

In [ ]:
plt.plot(ks, sils, 'o-', color='#7ee787', label='avg silhouette')
plt.xlabel('number of clusters k')
plt.ylabel('average silhouette score')
plt.title('Wine: average silhouette vs k (peak = true number of cultivars)')
plt.legend()
plt.show()